
### Load Feature Dataset


In [15]:

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer

# Locate the project directory
project_root = Path.cwd()

if not (project_root / "data/features/session_features.csv").exists():
    project_root = project_root.parent

# Load the extracted ML features
data_path = project_root / "data/features/session_features.csv"
df = pd.read_csv(data_path)

# Identify the 15 sensor features
feature_columns = [
    col for col in df.columns
    if col.startswith((
        "HR_", "HRV_", "Temperature_",
        "Acceleration_", "Gyroscope_"
    ))
]

print("Total ML samples:", len(df))
print("Total features:", len(feature_columns))
print("Unique participants:", df["Participant"].nunique())
print("Missing feature values:", df[feature_columns].isna().sum().sum())

display(df.head())


Total ML samples: 334
Total features: 15
Unique participants: 38
Missing feature values: 15


,Task,Participant,Session,HR_mean,HR_median,HR_std,HRV_mean,HRV_median,HRV_std,Temperature_mean,Temperature_median,Temperature_std,Acceleration_mean,Acceleration_median,Acceleration_std,Gyroscope_mean,Gyroscope_median,Gyroscope_std,Final_Fatigue_Score,Fatigue_Level
0,Composite,P1,Rep1,130.933060,131.60,5.368690,23.577235,20.575751,10.887614,34.713883,34.681758,0.289609,0.982510,0.981482,0.124014,7.569048,4.819670,7.645788,3,Low
1,Composite,P1,Rep2,137.085352,138.00,4.807611,12.932734,12.488656,6.593099,35.834101,35.813792,0.290845,0.983967,0.978990,0.111977,7.809359,5.294218,7.535942,5,Moderate
2,Composite,P1,Rep3,138.416223,139.75,5.775938,17.844573,14.910803,6.429464,36.789540,36.755521,0.099439,0.986493,0.979032,0.123755,8.464090,6.022367,7.967521,6,Moderate
3,Composite,P1,Rep4,139.749239,140.30,3.894888,20.843827,17.715367,12.270275,37.133465,37.119516,0.045261,0.979235,0.975083,0.126438,8.765335,5.985651,8.294293,7,High
4,Composite,P1,Rep5,138.760976,139.60,5.279229,13.741781,14.770766,5.037573,37.247084,37.244014,0.017937,0.995320,0.980618,0.153664,9.932089,7.417374,9.636959,7,High



### Split Training and Testing Data


In [16]:

from sklearn.model_selection import GroupShuffleSplit

# Define input features, target labels, and participant groups
X = df[feature_columns]
y = df["Fatigue_Level"]
groups = df["Participant"]

# Split approximately 80% for training and 20% for testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Create training and testing datasets
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Verify participant separation
train_participants = set(groups.iloc[train_idx])
test_participants = set(groups.iloc[test_idx])

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training participants:", len(train_participants))
print("Testing participants:", len(test_participants))
print("Shared participants:", len(train_participants & test_participants))

print("\nTraining Fatigue Distribution:")
print(y_train.value_counts())

print("\nTesting Fatigue Distribution:")
print(y_test.value_counts())


Training samples: 264
Testing samples: 70
Training participants: 30
Testing participants: 8
Shared participants: 0

Training Fatigue Distribution:
Fatigue_Level
Moderate    119
Low          75
High         70
Name: count, dtype: int64

Testing Fatigue Distribution:
Fatigue_Level
Low         27
Moderate    26
High        17
Name: count, dtype: int64



### Handle Missing Values Using Median Imputation


In [17]:

from sklearn.impute import SimpleImputer

# Check missing values before imputation
print("Training missing values:", X_train.isna().sum().sum())
print("Testing missing values:", X_test.isna().sum().sum())

# Create a median imputer
imputer = SimpleImputer(strategy="median")

# Learn replacement values from training data only
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_columns,
    index=X_train.index
)

# Apply the same training medians to testing data
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_columns,
    index=X_test.index
)

# Verify missing values were handled
print("\nAfter Imputation:")
print("Training missing values:", X_train_imputed.isna().sum().sum())
print("Testing missing values:", X_test_imputed.isna().sum().sum())

# Show the calculated replacement values
imputation_values = pd.DataFrame({
    "Feature": feature_columns,
    "Training Median": imputer.statistics_
})

display(imputation_values)


Training missing values: 15
Testing missing values: 0

After Imputation:
Training missing values: 0
Testing missing values: 0


,Feature,Training Median
0,HR_mean,106.519681
1,HR_median,106.400000
2,HR_std,6.198732
3,HRV_mean,62.290187
4,HRV_median,60.337622
5,HRV_std,21.197902
6,Temperature_mean,34.584877
7,Temperature_median,34.582265
8,Temperature_std,0.122800
9,Acceleration_mean,1.032630



### Train Random Forest Model


In [18]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Create a pipeline that handles missing values and trains the model
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Train using training participants only
rf_pipeline.fit(X_train, y_train)

print("Random Forest training completed!")
print("Training samples:", len(X_train))
print("Testing samples reserved:", len(X_test))
print("Input features:", len(feature_columns))
print("Target classes:", list(rf_pipeline.classes_))


Random Forest training completed!
Training samples: 264
Testing samples reserved: 70
Input features: 15
Target classes: ['High', 'Low', 'Moderate']



### Evaluate Random Forest Predictions


In [19]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

# Predict fatigue levels for unseen participants
y_pred = rf_pipeline.predict(X_test)

# Calculate model performance metrics
accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {accuracy:.2%}")
print(f"Balanced Accuracy: {balanced_accuracy:.2%}")
print(f"Macro F1-score: {macro_f1:.4f}")

# Show performance separately for each fatigue level
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=["Low", "Moderate", "High"],
    zero_division=0
))


Accuracy: 37.14%
Balanced Accuracy: 38.43%
Macro F1-score: 0.3705

Classification Report:
              precision    recall  f1-score   support

         Low       0.64      0.26      0.37        27
    Moderate       0.34      0.42      0.38        26
        High       0.30      0.47      0.36        17

    accuracy                           0.37        70
   macro avg       0.43      0.38      0.37        70
weighted avg       0.45      0.37      0.37        70




### Compare Random Forest with Baseline


In [20]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Create a baseline that predicts the most common training class
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Predict fatigue levels for test participants
baseline_pred = baseline.predict(X_test)

# Compare both models on the same test data
comparison = pd.DataFrame({
    "Model": ["Majority Baseline", "Random Forest"],
    "Accuracy (%)": [
        accuracy_score(y_test, baseline_pred) * 100,
        accuracy_score(y_test, y_pred) * 100
    ],
    "Balanced Accuracy (%)": [
        balanced_accuracy_score(y_test, baseline_pred) * 100,
        balanced_accuracy_score(y_test, y_pred) * 100
    ],
    "Macro F1": [
        f1_score(y_test, baseline_pred, average="macro"),
        f1_score(y_test, y_pred, average="macro")
    ]
})

display(comparison.round(2))


,Model,Accuracy (%),Balanced Accuracy (%),Macro F1
0,Majority Baseline,37.14,33.33,0.18
1,Random Forest,37.14,38.43,0.37



### Participant-Independent Cross-Validation


In [21]:

from sklearn.model_selection import GroupKFold, cross_validate

# Identify participants belonging to the training set
train_groups = df.loc[X_train.index, "Participant"]

# Create five participant-independent validation folds
cv = GroupKFold(n_splits=5)

# Evaluate the existing Random Forest pipeline
cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    groups=train_groups,
    cv=cv,
    scoring={
        "Accuracy": "accuracy",
        "Balanced Accuracy": "balanced_accuracy",
        "Macro F1": "f1_macro"
    },
    n_jobs=-1
)

# Show performance for each validation fold
cv_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": cv_results["test_Accuracy"],
    "Balanced Accuracy": cv_results["test_Balanced Accuracy"],
    "Macro F1": cv_results["test_Macro F1"]
})

display(cv_summary.round(4))

print("\nAverage Cross-Validation Results:")
print(cv_summary.iloc[:, 1:].mean().round(4))


,Fold,Accuracy,Balanced Accuracy,Macro F1
0,1,0.2000,0.1919,0.1907
1,2,0.3818,0.4140,0.3854
2,3,0.4444,0.5038,0.3972
3,4,0.2000,0.2281,0.1827
4,5,0.3000,0.3042,0.3008



Average Cross-Validation Results:
Accuracy             0.3053
Balanced Accuracy    0.3284
Macro F1             0.2914
dtype: float64



### Prepare Time-Series Features for XGBoost


In [29]:

# Define five chest-related sensor inputs
sensor_columns = [
    "HR_Processed",
    "HRV",
    "Temperature",
    "Acceleration_Magnitude",
    "Gyroscope_Proxy"
]

time_points = 32
records = []

# Process each participant's activity session separately
for (task, participant, session), data in df.groupby(
    ["Task", "Participant", "Session"]
):

    # Keep sensor readings in chronological order
    data = data.sort_values("Elapsed_Time_s")

    # Select 32 actual readings distributed across the session
    positions = np.linspace(
        0, len(data) - 1, time_points
    ).round().astype(int)

    selected = data.iloc[positions]

    # Use only the final fatigue score as the target
    final_score = data["Physical Fatigue - Final"].iloc[0]

    if final_score <= 3:
        label = "Low"
    elif final_score <= 6:
        label = "Moderate"
    else:
        label = "High"

    record = {
        "Task": task,
        "Participant": participant,
        "Session": session,
        "Fatigue_Level": label
    }

    # Preserve individual sensor readings, not statistics
    for sensor in sensor_columns:
        for t, value in enumerate(selected[sensor].to_numpy()):
            record[f"{sensor}_t{t:02d}"] = value

    records.append(record)

# Create the XGBoost feature dataset
raw_features_df = pd.DataFrame(records)

raw_feature_columns = [
    col for col in raw_features_df.columns
    if col not in [
        "Task", "Participant", "Session", "Fatigue_Level"
    ]
]

print("Total ML samples:", len(raw_features_df))
print("Time points per session:", time_points)
print("Sensor inputs:", len(sensor_columns))
print("Total features:", len(raw_feature_columns))
print("Missing values:", raw_features_df[raw_feature_columns].isna().sum().sum())

print("\nFatigue Distribution:")
print(raw_features_df["Fatigue_Level"].value_counts())

display(raw_features_df.iloc[:5, :9])


Total ML samples: 334
Time points per session: 32
Sensor inputs: 5
Total features: 160
Missing values: 2181

Fatigue Distribution:
Fatigue_Level
Moderate    145
Low         102
High         87
Name: count, dtype: int64


,Task,Participant,Session,Fatigue_Level,HR_Processed_t00,HR_Processed_t01,HR_Processed_t02,HR_Processed_t03,HR_Processed_t04
0,Composite,P1,Rep1,Low,102.5,NaN,NaN,125.2,129.2
1,Composite,P1,Rep2,Moderate,129.2,127.2,133.2,131.7,129.8
2,Composite,P1,Rep3,Moderate,122.9,128.1,131.8,132.6,131.3
3,Composite,P1,Rep4,High,132.3,131.5,140.7,137.8,137.6
4,Composite,P1,Rep5,High,123.4,127.8,128.6,132.4,132.5



### Train XGBoost Using Time-Ordered Sensor Readings


In [30]:

from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
import pandas as pd

# Separate sensor readings, fatigue labels, and participants
X_raw = raw_features_df[raw_feature_columns]
y_raw = raw_features_df["Fatigue_Level"]
groups_raw = raw_features_df["Participant"]

# Split by participant to prevent data leakage
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_raw, test_idx_raw = next(
    splitter.split(X_raw, y_raw, groups=groups_raw)
)

X_train_raw = X_raw.iloc[train_idx_raw]
X_test_raw = X_raw.iloc[test_idx_raw]

# Encode fatigue labels as numerical classes
encoder_raw = LabelEncoder()
y_train_raw = encoder_raw.fit_transform(
    y_raw.iloc[train_idx_raw]
)

train_groups_raw = groups_raw.iloc[train_idx_raw]

# Verify that training and testing participants are different
train_ids = set(groups_raw.iloc[train_idx_raw])
test_ids = set(groups_raw.iloc[test_idx_raw])

assert not (train_ids & test_ids)

# Create XGBoost model using actual time-ordered readings
# XGBoost handles missing values internally
xgb_raw_model = XGBClassifier(
    objective="multi:softprob",
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=4
)

# Evaluate using unseen participants within the training set
cv_results_raw = cross_validate(
    xgb_raw_model,
    X_train_raw,
    y_train_raw,
    groups=train_groups_raw,
    cv=GroupKFold(n_splits=5),
    scoring={
        "Accuracy": "accuracy",
        "Balanced Accuracy": "balanced_accuracy",
        "Macro F1": "f1_macro"
    },
    n_jobs=1
)

# Display cross-validation performance
results_raw = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": cv_results_raw["test_Accuracy"],
    "Balanced Accuracy": cv_results_raw["test_Balanced Accuracy"],
    "Macro F1": cv_results_raw["test_Macro F1"]
})

display(results_raw.round(4))

print("\nAverage Results:")
print(results_raw.iloc[:, 1:].mean().round(4))

# Train final model on all training participants
xgb_raw_model.fit(X_train_raw, y_train_raw)

print("\nXGBoost time-series training completed!")
print("Training samples:", len(X_train_raw))
print("Reserved test samples:", len(X_test_raw))
print("Features:", len(raw_feature_columns))
print("Shared participants:", len(train_ids & test_ids))


,Fold,Accuracy,Balanced Accuracy,Macro F1
0,1,0.4364,0.4200,0.3479
1,2,0.4909,0.4207,0.4335
2,3,0.3519,0.3663,0.2964
3,4,0.1800,0.2085,0.1579
4,5,0.2200,0.2354,0.2089



Average Results:
Accuracy             0.3358
Balanced Accuracy    0.3302
Macro F1             0.2889
dtype: float64

XGBoost time-series training completed!
Training samples: 264
Reserved test samples: 70
Features: 160
Shared participants: 0



## Prepare Row-Level Sensor Data for XGBoost


In [31]:

# Define our five sensor inputs
sensor_columns = [
    "HR_Processed",
    "HRV",
    "Temperature",
    "Acceleration_Magnitude",
    "Gyroscope_Proxy"
]

# Keep sensor readings, time, participant information, and final fatigue
row_df = df[
    [
        "Task",
        "Participant",
        "Session",
        "Elapsed_Time_s",
        *sensor_columns,
        "Physical Fatigue - Final"
    ]
].copy()

# Convert the final fatigue score into Low, Moderate, High
row_df["Fatigue_Level"] = pd.cut(
    row_df["Physical Fatigue - Final"],
    bins=[-1, 3, 6, 10],
    labels=["Low", "Moderate", "High"]
)

# Verify each session has one final fatigue label
assert (
    row_df.groupby(["Task", "Participant", "Session"])[
        "Fatigue_Level"
    ].nunique() == 1
).all()

# Check the prepared dataset
print("Total sensor rows:", len(row_df))
print("Sensor inputs:", len(sensor_columns))
print("Unique participants:", row_df["Participant"].nunique())

print(
    "Activity sessions:",
    row_df[["Task", "Participant", "Session"]]
    .drop_duplicates().shape[0]
)

print("\nMissing sensor values:")
print(row_df[sensor_columns].isna().sum())

display(row_df.head(10))


Total sensor rows: 286838
Sensor inputs: 5
Unique participants: 38
Activity sessions: 334

Missing sensor values:
HR_Processed              52584
HRV                        4836
Temperature                   0
Acceleration_Magnitude        0
Gyroscope_Proxy               0
dtype: int64


,Task,Participant,Session,Elapsed_Time_s,HR_Processed,HRV,Temperature,Acceleration_Magnitude,Gyroscope_Proxy,Physical Fatigue - Final,Fatigue_Level
0,Composite,P1,Rep1,0.0,102.5,22.402814,34.224213,0.137257,14.436704,3,Low
1,Composite,P1,Rep1,0.5,104.7,22.376775,34.226817,2.341925,1.233281,3,Low
2,Composite,P1,Rep1,1.0,108.2,22.351200,34.228885,2.613419,13.288418,3,Low
3,Composite,P1,Rep1,1.5,108.2,22.326121,34.230427,0.048813,34.480159,3,Low
4,Composite,P1,Rep1,2.0,108.2,22.301569,34.231472,0.326829,5.812640,3,Low
5,Composite,P1,Rep1,2.5,108.2,22.277574,34.232053,0.959307,16.733941,3,Low
6,Composite,P1,Rep1,3.0,108.2,22.254168,34.232200,0.969873,6.454106,3,Low
7,Composite,P1,Rep1,3.5,108.2,22.231382,34.231944,0.979266,3.563215,3,Low
8,Composite,P1,Rep1,4.0,108.2,22.209245,34.231317,1.054481,16.625290,3,Low
9,Composite,P1,Rep1,4.5,108.2,22.187790,34.230351,0.943052,5.428093,3,Low



### Split Row-Level Training and Testing Data


In [32]:

from sklearn.model_selection import GroupShuffleSplit

# Use five sensor measurements and elapsed time as inputs
row_input_columns = sensor_columns + ["Elapsed_Time_s"]

X_rows = row_df[row_input_columns]
y_rows = row_df["Fatigue_Level"]
participant_groups = row_df["Participant"]

# Split participants, not individual sensor rows
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_rows, test_idx_rows = next(
    splitter.split(X_rows, y_rows, groups=participant_groups)
)

# Prepare training and testing datasets
X_train_rows = X_rows.iloc[train_idx_rows].copy()
X_test_rows = X_rows.iloc[test_idx_rows].copy()

y_train_rows = y_rows.iloc[train_idx_rows].copy()
y_test_rows = y_rows.iloc[test_idx_rows].copy()

# Preserve session information for final evaluation
train_info = row_df.iloc[train_idx_rows][
    ["Task", "Participant", "Session"]
].copy()

test_info = row_df.iloc[test_idx_rows][
    ["Task", "Participant", "Session"]
].copy()

# Verify participant-independent splitting
train_participants = set(train_info["Participant"])
test_participants = set(test_info["Participant"])

print("Training rows:", len(X_train_rows))
print("Testing rows:", len(X_test_rows))
print("Training participants:", len(train_participants))
print("Testing participants:", len(test_participants))
print("Shared participants:", len(train_participants & test_participants))

print("\nTraining sessions:", len(train_info.drop_duplicates()))
print("Testing sessions:", len(test_info.drop_duplicates()))


Training rows: 228159
Testing rows: 58679
Training participants: 30
Testing participants: 8
Shared participants: 0

Training sessions: 264
Testing sessions: 70



### Train Row-Level XGBoost Model


In [33]:

import numpy as np
from xgboost import XGBClassifier

# Convert fatigue labels into numerical classes
label_mapping = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}

y_train_encoded_rows = (
    y_train_rows.map(label_mapping).astype(int)
)

# Count readings in each training session
session_columns = ["Task", "Participant", "Session"]

session_sizes = train_info.groupby(
    session_columns
)["Session"].transform("size")

# Give every session equal total training weight
n_train_sessions = len(train_info.drop_duplicates())

sample_weights = (
    len(train_info) / n_train_sessions
) / session_sizes.to_numpy()

# Build the XGBoost classifier
xgb_row_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=4
)

# Train using individual sensor readings
xgb_row_model.fit(
    X_train_rows,
    y_train_encoded_rows,
    sample_weight=sample_weights
)

print("Row-level XGBoost training completed!")
print("Training rows:", len(X_train_rows))
print("Training sessions:", n_train_sessions)
print("Input features:", X_train_rows.shape[1])
print("Classes:", label_mapping)


Row-level XGBoost training completed!
Training rows: 228159
Training sessions: 264
Input features: 6
Classes: {'Low': 0, 'Moderate': 1, 'High': 2}



### Evaluate Row-Level XGBoost


In [34]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

# Predict class probabilities for every test sensor reading
row_probabilities = xgb_row_model.predict_proba(X_test_rows)

# Define the order of fatigue classes
class_names = ["Low", "Moderate", "High"]
session_columns = ["Task", "Participant", "Session"]

# Combine predictions with their session identifiers
predictions_df = test_info.reset_index(drop=True).copy()
predictions_df["Actual"] = y_test_rows.astype(str).to_numpy()

for i, label in enumerate(class_names):
    predictions_df[f"Prob_{label}"] = row_probabilities[:, i]

# Verify that each session has one final fatigue label
assert (
    predictions_df.groupby(session_columns)["Actual"].nunique() == 1
).all()

# Average prediction probabilities within each session
session_results = predictions_df.groupby(
    session_columns, as_index=False
).agg(
    Actual=("Actual", "first"),
    Prob_Low=("Prob_Low", "mean"),
    Prob_Moderate=("Prob_Moderate", "mean"),
    Prob_High=("Prob_High", "mean")
)

# Select the fatigue class with the highest average probability
probability_columns = [
    "Prob_Low", "Prob_Moderate", "Prob_High"
]

session_results["Predicted"] = np.array(class_names)[
    session_results[probability_columns].to_numpy().argmax(axis=1)
]

# Evaluate predictions using sessions, not individual rows
y_true = session_results["Actual"]
y_pred_session = session_results["Predicted"]

print("Total evaluated sessions:", len(session_results))
print(f"Accuracy: {accuracy_score(y_true, y_pred_session):.2%}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred_session):.2%}")
print(f"Macro F1: {f1_score(y_true, y_pred_session, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred_session,
    labels=class_names,
    zero_division=0
))

display(session_results.head(10))


Total evaluated sessions: 70
Accuracy: 38.57%
Balanced Accuracy: 35.78%
Macro F1: 0.3055

Classification Report:
              precision    recall  f1-score   support

         Low       0.40      0.15      0.22        27
    Moderate       0.38      0.81      0.52        26
        High       0.40      0.12      0.18        17

    accuracy                           0.39        70
   macro avg       0.39      0.36      0.31        70
weighted avg       0.39      0.39      0.32        70



,Task,Participant,Session,Actual,Prob_Low,Prob_Moderate,Prob_High,Predicted
0,Composite,P13,Rep1,Moderate,0.290073,0.469168,0.240760,Moderate
1,Composite,P13,Rep2,Moderate,0.403133,0.451201,0.145666,Moderate
2,Composite,P13,Rep3,Moderate,0.222334,0.662420,0.115245,Moderate
3,Composite,P13,Rep4,High,0.042832,0.901957,0.055211,Moderate
4,Composite,P13,Rep5,High,0.058125,0.861644,0.080231,Moderate
5,Composite,P21,Rep1,Low,0.382140,0.522451,0.095409,Moderate
6,Composite,P21,Rep2,Low,0.344796,0.468280,0.186924,Moderate
7,Composite,P21,Rep3,Moderate,0.264936,0.419007,0.316058,Moderate
8,Composite,P21,Rep4,Moderate,0.276436,0.383203,0.340361,Moderate
9,Composite,P21,Rep5,Moderate,0.252703,0.389848,0.357449,Moderate
